# <!-- TITLE --> [CETO1 V2] - Baseline ML avec détourage
<!-- DESC --> La baseline, avec des empreintes calculées sur les photos détourées par le `Segmenter` (CetaceaNet, MiewID, MegaDescriptor)

## Pourquoi cette baseline
Ce notebook montre ce qu'on obtient déjà **sans pipeline**, avec seulement les empreintes de référence et la bibliothèque `dorsalmatch`. C'est le niveau à battre : toute nouvelle donnée ou nouveau modèle se compare à ces chiffres.

## La question
L'historique, ce sont des photos **déjà identifiées** par Miraceti : chacune porte le `TUTR` de son dauphin. Le catalogue, c'est une photo par côté pour chaque dauphin connu.

**Si on cache les annotations de l'historique, est-ce que le ML retrouve le bon dauphin dans le catalogue ?** Comme les annotations existent, on peut noter cette reconstruction.

Pour chaque photo (puis pour chaque dauphin), le bon dauphin tombe dans un de ces trois cas :

| Cas | Ce que ça veut dire pour Miraceti |
|---|---|
| **bien rangé** : rang 1 | le premier candidat proposé est le bon |
| **dans le top 10, mal rangé** : rang 2 à 10 | le bon est dans la liste, mais il faut descendre |
| **hors du top 10** | le bon n'est pas dans ce que Miraceti regarde |

## Les données
Les empreintes et les photos prêtes se récupèrent avec `notebooks/data_livraison.ipynb`, dans la partie ML et MLOps (décommenter le téléchargement des photos pour le Step 5).

## Ce que change la V2

Le [notebook de détourage](detourage_segmenter.ipynb) montre que le `Segmenter` isole bien la nageoire. Cette V2 mesure si ça aide à retrouver le bon dauphin.

Chaque modèle est évalué sur les mêmes photos, dans cinq variantes :

| Variante | Fichier | Traitement |
|---|---|---|
| brut | `<modèle>.npy` | aucun (baseline) |
| détouré recadré | `<modèle>_detoure.npy` | fond blanc, recadré sur le dauphin (réglages par défaut de [`segmenter.yaml`](../dorsalmatch/segmenter.yaml)) |
| fond blanc, sans recadrage | `<modèle>_fond_blanc.npy` | fond blanc, cadre d'origine |
| fond blanc, marge 3 % | `<modèle>_fond_blanc_marge3.npy` | fond blanc, cadre d'origine, marge autour du masque de 3 % au lieu de 0,5 % |
| recadré, mer gardée | `<modèle>_recadre_mer.npy` | recadré en carré autour du dauphin (marge 10 %), fond conservé |

Calcul des empreintes (chaque photo est préparée une fois pour les trois modèles) :

```bash
uv run --group ml python -m pipeline.embeddings cetaceanet miewid megadescriptor --detoure --kinds catalogue historique
uv run --group ml python -m pipeline.embeddings cetaceanet miewid megadescriptor --detoure --segmenter crop=false --suffixe _fond_blanc --kinds catalogue historique
uv run --group ml python -m pipeline.embeddings cetaceanet miewid megadescriptor --detoure --segmenter crop=false margin=0.03 --suffixe _fond_blanc_marge3 --kinds catalogue historique
uv run --group ml python -m pipeline.embeddings cetaceanet miewid megadescriptor --detoure --mer --suffixe _recadre_mer --kinds catalogue historique
```

Le Step 6 compare chaque variante à la photo brute.

## Step 1 - Initialisation

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import dorsalmatch  # noqa: E402

# CONSTANTES
DATA = ROOT / "data"
HISTORIQUES = ["2026-10-29_miraceti", "2026-11-06_bestof_20230307_sete"]
CATALOGUE = "2026-11-06_catalogue_tt"
TOP = 10
SIDES = ("D", "G")
BASE_MODELS = ["cetaceanet", "miewid", "megadescriptor"]
# Suffixe des empreintes -> nom de la variante
VARIANTS = {
    "": "brut",
    "_detoure": "détouré recadré",
    "_fond_blanc": "fond blanc, sans recadrage",
    "_fond_blanc_marge3": "fond blanc, marge 3 %",
    "_recadre_mer": "recadré, mer gardée",
}
# Le premier (cetaceanet brut) choisit les photos mises de côté (Step 2.1)
MODELS_TO_COMPARE = [model + suffix for model in BASE_MODELS for suffix in VARIANTS]
NEAR_IDENTICAL = 0.97  # au-delà, c'est la même photo.
CASES = ["bien rangé", f"top {TOP}, mal rangé", f"hors top {TOP}"]

## Step 2 - Les données

In [ ]:
def load(kind, delivery, model, data=DATA):
    folder = data / "derived" / "embeddings" / kind / delivery
    # keep_default_na=False : un côté vide reste "", pas NaN
    photos = pd.read_csv(folder / "photos.csv", dtype=str, keep_default_na=False)
    return photos.to_dict("records"), np.load(folder / f"{model}.npy")


# Toutes les photos contre un côté du catalogue : le meilleur cosinus de chaque dauphin
def best_cosines(photos, embeddings, catalogue, catalogue_embeddings, side):
    lines = [line for line, photo in enumerate(photos) if photo["cote"] in SIDES]
    catalogue_lines = [
        line for line, photo in enumerate(catalogue) if photo["cote"] == side
    ]
    if not lines or not catalogue_lines:
        return {}, set()

    best, identical = {}, set()
    results = dorsalmatch.search(
        embeddings[lines], catalogue_embeddings[catalogue_lines], k=len(catalogue_lines)
    )
    for closest in results:
        for position, cosine in closest:
            individual = catalogue[catalogue_lines[position]]["individu"]
            if cosine >= NEAR_IDENTICAL:
                identical.add(individual)
                continue
            best[individual] = max(best.get(individual, -1.0), cosine)
    return best, identical


# Score d'un dauphin : le meilleur cosinus, tous côtés confondus
def rank(photos, embeddings, catalogue, catalogue_embeddings):
    by_side, identical = {}, set()
    for side in SIDES:
        by_side[side], found = best_cosines(
            photos, embeddings, catalogue, catalogue_embeddings, side
        )
        identical |= found

    candidates = []
    for individual in set(by_side["D"]) | set(by_side["G"]):
        scores = {
            side: by_side[side][individual]
            for side in SIDES
            if individual in by_side[side]
        }
        candidates.append(
            {"individu": individual, "score": float(max(scores.values())), **scores}
        )
    return sorted(
        candidates, key=lambda candidate: (-candidate["score"], candidate["individu"])
    ), identical

In [ ]:
def load_all(model, data=DATA):
    history, history_embeddings = [], []
    for delivery in HISTORIQUES:
        photos, embeddings = load("historique", delivery, model, data)
        history += [{**photo, "livraison": delivery} for photo in photos]
        history_embeddings.append(embeddings)
    catalogue, catalogue_embeddings = load("catalogue", CATALOGUE, model, data)
    return history, np.vstack(history_embeddings), catalogue, catalogue_embeddings


DATA_BY_MODEL = {model: load_all(model) for model in MODELS_TO_COMPARE}

history, _, catalogue, _ = DATA_BY_MODEL[MODELS_TO_COMPARE[0]]
print(
    f"Historique : {len(history)} photos, "
    f"{len({photo['individu'] for photo in history})} dauphins"
)
# Sans la photo au nom illisible : elle n'a pas de côté, ce n'est pas un dauphin
print(
    f"Catalogue  : {len(catalogue)} photos, "
    f"{len({photo['individu'] for photo in catalogue if photo['cote']})} dauphins"
)
print(
    "Dimensions des embeddings : "
    + ", ".join(
        f"{model}={history_embeddings.shape[1]}"
        for model, (_, history_embeddings, _, _) in DATA_BY_MODEL.items()
    )
)

### 2.1 - Les photos qu'on ne peut pas noter

Le catalogue a été construit à partir de l'historique : beaucoup de photos de l'historique **sont** la photo du catalogue. Le ML retrouverait sa propre photo, ce qui ne prouve rien.

Les paires de photos presque identiques (cosinus ≥ `NEAR_IDENTICAL`) ne comptent donc pas dans le score. Une photo de l'historique dont la seule photo au catalogue, du même côté, est elle-même n'a rien à retrouver : elle est mise de côté.

In [ ]:
# Le seuil ne dépend pas du modèle.
# On met de côté avec le modèle de référence une seule fois.
REFERENCE = MODELS_TO_COMPARE[0]
history, history_embeddings, catalogue, catalogue_embeddings = DATA_BY_MODEL[REFERENCE]


def findable(line, near_identical=NEAR_IDENTICAL):
    photo = history[line]
    same = [
        index
        for index, other in enumerate(catalogue)
        if other["individu"] == photo["individu"] and other["cote"] == photo["cote"]
    ]
    if not same:
        return "côté absent du catalogue"
    if (catalogue_embeddings[same] @ history_embeddings[line] >= near_identical).all():
        return "c'est la photo du catalogue"
    return None


reasons = [findable(line) for line in range(len(history))]
REASONS = dict.fromkeys(MODELS_TO_COMPARE, reasons)  # la même liste pour tous

for reason in sorted({reason for reason in reasons if reason}):
    print(f"{reasons.count(reason):>4} photos mises de côté : {reason}")
print(f"{reasons.count(None):>4} photos notées")

## Step 3 - La reconstruction

Pour chaque photo notée, le ML classe les dauphins du catalogue du plus au moins ressemblant, avec `dorsalmatch.search` et la fonction `rank` du Step 2, **comme dans la démo** : tous côtés confondus, meilleur cosinus par dauphin.

Puis, **par dauphin** : toutes ses photos de l'historique ensemble, les deux côtés combinés (meilleur cosinus), comme ce que Miraceti voit pour un individu d'un best-of.

In [ ]:
def reconstruct(individual, lines):
    candidates, _ = rank(
        [history[line] for line in lines],
        history_embeddings[lines],
        catalogue,
        catalogue_embeddings,
    )
    names = [candidate["individu"] for candidate in candidates]
    position = names.index(individual) if individual in names else None
    return {
        "individu": individual,
        "lines": lines,
        "candidats": candidates,
        "rang": position + 1 if position is not None else None,
        "score_bon": candidates[position]["score"] if position is not None else None,
        "score_premier_faux": next(
            candidate["score"]
            for candidate in candidates
            if candidate["individu"] != individual
        ),
    }


def case(result):
    if result["rang"] == 1:
        return CASES[0]
    if result["rang"] and result["rang"] <= TOP:
        return CASES[1]
    return CASES[2]


BY_PHOTO, BY_DOLPHIN = {}, {}

for model in MODELS_TO_COMPARE:
    # Les deux fonctions ci-dessus lisent ces noms.
    # On les pointe sur le modèle en cours.
    history, history_embeddings, catalogue, catalogue_embeddings = DATA_BY_MODEL[model]
    reasons = REASONS[model]

    by_photo = [
        reconstruct(history[line]["individu"], [line])
        for line in range(len(history))
        if reasons[line] is None
    ]

    lines_by_dolphin = {}
    for line in range(len(history)):
        if reasons[line] is None:
            lines_by_dolphin.setdefault(history[line]["individu"], []).append(line)
    by_dolphin = [
        reconstruct(individual, lines)
        for individual, lines in sorted(lines_by_dolphin.items())
    ]

    BY_PHOTO[model], BY_DOLPHIN[model] = by_photo, by_dolphin
    print(
        f"{model:<12} {len(by_photo)} photos et {len(by_dolphin)} dauphins reconstruits"
    )

## Step 4 - Quanti

### 4.1 - Combien de bien rangés, de mal rangés, de ratés ?

In [ ]:
def counts(results, title):
    print(title)
    for name in CASES:
        number = sum(case(result) == name for result in results)
        print(f"  {name:<22} {number:>4}  ({number / len(results):.0%})")


for model in MODELS_TO_COMPARE:
    counts(BY_PHOTO[model], f"{model} · par photo ({len(BY_PHOTO[model])})")
    counts(BY_DOLPHIN[model], f"{model} · par dauphin ({len(BY_DOLPHIN[model])})")

In [ ]:
colors = ["#2e7d32", "#f9a825", "#c62828"]  # bien rangé, mal rangé, hors top
positions = np.arange(len(MODELS_TO_COMPARE))
width = 0.8 / len(CASES)

fig, axes = plt.subplots(1, 2, figsize=(26, 5.2))
for ax, (per_model, title) in zip(
    axes, [(BY_PHOTO, "par photo"), (BY_DOLPHIN, "par dauphin")], strict=True
):
    biggest = 0
    for offset, (name, color) in enumerate(zip(CASES, colors, strict=True)):
        numbers = [
            sum(case(result) == name for result in per_model[model])
            for model in MODELS_TO_COMPARE
        ]
        biggest = max(biggest, *numbers)
        bars = ax.bar(
            positions + (offset - (len(CASES) - 1) / 2) * width,
            numbers,
            width,
            color=color,
            label=name,
        )
        ax.bar_label(
            bars,
            [
                f"{number}\n{number / len(per_model[model]):.0%}"
                for number, model in zip(numbers, MODELS_TO_COMPARE, strict=True)
            ],
            fontsize=8,
            padding=2,
        )
    ax.set_xticks(positions, MODELS_TO_COMPARE, rotation=25, ha="right")
    ax.set_ylim(0, biggest * 1.3)  # la place des étiquettes sur deux lignes
    ax.set_title(title)

axes[0].legend()
plt.tight_layout()
plt.show()

### 4.2 - Quand il est hors du top 10, à quel rang est-il ?

Juste après la liste, ou complètement perdu ?

In [ ]:
for model in MODELS_TO_COMPARE:
    missed = [result for result in BY_PHOTO[model] if case(result) == CASES[2]]
    print(f"{model} · {len(missed)} photos hors du top {TOP} :")
    print(
        "  rang médian du bon dauphin : "
        f"{np.median([result['rang'] for result in missed]):.0f}"
    )
    for limit in (20, 50, 100):
        print(
            f"  dans le top {limit:<4} "
            f"{sum(result['rang'] <= limit for result in missed)}"
        )

### 4.3 - Les scores

Le score est le meilleur cosinus entre les empreintes des dorsales, tous côtés confondus : plus il est proche de 1, plus les dorsales se ressemblent.

Pour chaque photo : le score du **bon dauphin**, et celui du **premier faux candidat**. Si les deux se chevauchent, aucun seuil ne peut dire « c'est lui » : c'est à Miraceti de regarder.

In [ ]:
bins = np.linspace(0.2, 1, 33)
fig, axes = plt.subplots(
    len(MODELS_TO_COMPARE),
    len(CASES),
    figsize=(15, 3.5 * len(MODELS_TO_COMPARE)),
    sharey=True,
)
for row, model in enumerate(MODELS_TO_COMPARE):
    for column, name in enumerate(CASES):
        ax = axes[row, column]
        results = [result for result in BY_PHOTO[model] if case(result) == name]
        ax.hist(
            [result["score_bon"] for result in results],
            bins=bins,
            alpha=0.6,
            label="bon dauphin",
        )
        ax.hist(
            [result["score_premier_faux"] for result in results],
            bins=bins,
            alpha=0.6,
            label="premier faux",
        )
        ax.set_title(f"{model} · {name} ({len(results)} photos)")
        ax.set_xlabel("score")
    axes[row, 0].set_ylabel("photos")

axes[0, 0].legend()
plt.tight_layout()
plt.show()

for model in MODELS_TO_COMPARE:
    print(model)
    for name in CASES:
        results = [result for result in BY_PHOTO[model] if case(result) == name]
        print(
            f"  {name:<22} score du bon : médiane "
            f"{np.median([r['score_bon'] for r in results]):.2f}   "
            "premier faux : médiane "
            f"{np.median([r['score_premier_faux'] for r in results]):.2f}"
        )

### 4.4 - Droite ou gauche ?

In [ ]:
for model in MODELS_TO_COMPARE:
    print(model)
    history = DATA_BY_MODEL[model][0]
    for side, name in (("D", "droite"), ("G", "gauche")):
        results = [
            result
            for result in BY_PHOTO[model]
            if history[result["lines"][0]]["cote"] == side
        ]
        shares = "   ".join(
            f"{case_name} "
            f"{sum(case(r) == case_name for r in results) / len(results):.0%}"
            for case_name in CASES
        )
        print(f"  {name:<7} {len(results):>4} photos   {shares}")

## Step 5 - Quali

Une photo de l'historique, face aux 10 dauphins proposés par le ML : **vert**, le bon dauphin ; **rouge**, un autre. Le chiffre est le score.

Pour chaque cas, regarder ce qui se passe : photo floue, dorsale de loin, angle, dorsale peu marquée…

**Les photos de Miraceti ne sont pas publiques : effacer les sorties de cette partie avant de commiter ce notebook.** Les chiffres et les graphiques des parties précédentes peuvent rester.

In [ ]:
def photo_path(kind, delivery, photo):
    return (
        DATA
        / "curated"
        / kind
        / "photos"
        / delivery
        / photo["individu"]
        / photo["fichier_origine"]
    )


SIDES_NAMES = {"D": "droite", "G": "gauche"}
# Case grise : pas de photo pour ce côté, comme dans la démo
MISSING = np.full((260, 400, 3), 225, dtype=np.uint8)


def catalogue_photo(individual, side):
    return next(
        (
            photo
            for photo in catalogue
            if photo["individu"] == individual and photo["cote"] == side
        ),
        None,
    )


def small(path):
    image = Image.open(path).convert("RGB")
    image.thumbnail((400, 400))
    return image


def show(result, prefix=""):
    query = history[result["lines"][0]]
    candidates = result["candidats"][:TOP]
    fig = plt.figure(figsize=(15, 9), dpi=80)
    grid = fig.add_gridspec(
        4,
        6,
        width_ratios=[2.4, 1, 1, 1, 1, 1],
        wspace=0.08,
        hspace=0.45,
        top=0.9,
        bottom=0.02,
    )
    ax = fig.add_subplot(grid[:, 0])
    ax.imshow(small(photo_path("historique", query["livraison"], query)))
    ax.set_title(
        f"{prefix}{query['individu']} · {query['cote']} · rang {result['rang']}",
        loc="left",
        fontsize=12,
    )
    ax.axis("off")
    for position, candidate in enumerate(candidates):
        good = candidate["individu"] == query["individu"]
        for side_line, side in enumerate(SIDES):
            ax = fig.add_subplot(
                grid[(position // 5) * 2 + side_line, 1 + position % 5]
            )
            photo = catalogue_photo(candidate["individu"], side)
            ax.imshow(
                small(photo_path("catalogue", CATALOGUE, photo)) if photo else MISSING
            )
            score = f"{candidate[side]:.2f}" if side in candidate else "–"
            label = (
                f"{SIDES_NAMES[side]} {score}"
                if photo
                else f"pas de photo {SIDES_NAMES[side]}"
            )
            title = (
                f"{position + 1}. {candidate['individu']} · "
                f"{candidate['score']:.2f}\n{label}"
                if side_line == 0
                else label
            )
            ax.set_title(title, fontsize=9, color="green" if good else "red")
            ax.axis("off")
    plt.show()


RANDOM = np.random.default_rng(0)


# La même photo chez deux modèles : la même ligne d'historique
def by_line(model, name):
    return {
        result["lines"][0]: result for result in BY_PHOTO[model] if case(result) == name
    }


def examples(name, number=3, models=MODELS_TO_COMPARE):
    common = set.intersection(*(set(by_line(model, name)) for model in models))
    for line in RANDOM.choice(
        sorted(common), size=min(number, len(common)), replace=False
    ):
        for model in models:
            show(by_line(model, name)[line], f"{model} · ")

### 5.1 - Bien rangé

In [ ]:
examples(CASES[0], models=BASE_MODELS)

### 5.2 - Dans le top 10, mal rangé

In [ ]:
examples(CASES[1], models=BASE_MODELS)

### 5.3 - Hors du top 10

In [ ]:
examples(CASES[2], models=BASE_MODELS)

## Step 6 - Ce que le détourage change

Chaque variante comparée à la photo brute, sur les mêmes 363 photos. En plus des trois cas du Step 4 :

- **rang médian** du bon dauphin ;
- **écart** : score du bon dauphin moins score du premier faux (positif : le bon passe devant).

Lecture :

- fond blanc, sans recadrage aussi mauvais que détouré recadré : c'est retirer la mer qui dégrade ;
- fond blanc, marge 3 % meilleur que marge 0,5 % : la marge fine rognait la nageoire ;
- recadré, mer gardée meilleur que brut : agrandir le dauphin aide.

In [ ]:
PERCENT = {"bien rangé", f"top {TOP}", f"hors top {TOP}"}


def summary(results):
    ranks = np.array([r["rang"] if r["rang"] else 10_000 for r in results])
    gaps = np.array([(r["score_bon"] or 0) - r["score_premier_faux"] for r in results])
    return {
        "bien rangé": (ranks == 1).mean(),
        f"top {TOP}": (ranks <= TOP).mean(),
        f"hors top {TOP}": (ranks > TOP).mean(),
        "rang médian": float(np.median(ranks)),
        "écart médian": float(np.median(gaps)),
    }


def fmt(name, value, signed=False):
    sign = "+" if signed else ""
    if name in PERCENT:
        return f"{value:{sign}.0%}"
    return f"{value:{sign}.2f}" if "écart" in name else f"{value:{sign}.0f}"


for level, per_model in (("par photo", BY_PHOTO), ("par dauphin", BY_DOLPHIN)):
    rows = {}
    for model in BASE_MODELS:
        raw = summary(per_model[model])
        for suffix, variant in VARIANTS.items():
            values = summary(per_model[model + suffix])
            rows[(model, variant)] = {
                name: fmt(name, values[name])
                + (
                    ""
                    if not suffix
                    else f" ({fmt(name, values[name] - raw[name], signed=True)})"
                )
                for name in values
            }
    print(f"{level} · entre parenthèses : écart avec la photo brute")
    display(pd.DataFrame(rows).T)

### 6.1 - Photo par photo

Le cas de chaque photo, brute (lignes) et avec la variante (colonnes). Au-dessus de la diagonale, la variante dégrade ; en dessous, elle améliore.

In [ ]:
def cases_by_line(model):
    return {result["lines"][0]: case(result) for result in BY_PHOTO[model]}


TRANSITIONS = {}
for model in BASE_MODELS:
    for suffix, variant in list(VARIANTS.items())[1:]:
        changes = pd.DataFrame(
            {"brut": cases_by_line(model), variant: cases_by_line(model + suffix)}
        )
        TRANSITIONS[(model, suffix)] = changes
        won = ((changes["brut"] != CASES[0]) & (changes[variant] == CASES[0])).sum()
        lost = ((changes["brut"] == CASES[0]) & (changes[variant] != CASES[0])).sum()
        print(f"{model} · {variant} : rang 1 gagné sur {won} photos, perdu sur {lost}")
        display(
            pd.crosstab(changes["brut"], changes[variant]).reindex(
                index=CASES, columns=CASES, fill_value=0
            )
        )

### 6.2 - Écart de score

Écart entre le score du bon dauphin et celui du premier faux. Une variante qui aide déplace la distribution vers la droite.

In [ ]:
fig, axes = plt.subplots(1, len(BASE_MODELS), figsize=(15, 4), sharey=True)
bins = np.linspace(-0.4, 0.6, 41)
for ax, model in zip(axes, BASE_MODELS, strict=True):
    for suffix, variant in VARIANTS.items():
        gaps = [
            (r["score_bon"] or 0) - r["score_premier_faux"]
            for r in BY_PHOTO[model + suffix]
        ]
        ax.hist(gaps, bins=bins, alpha=0.45, label=variant, histtype="stepfilled")
    ax.axvline(0, color="grey", linestyle=":")
    ax.set_title(model)
    ax.set_xlabel("score du bon - score du premier faux")
axes[0].set_ylabel("photos")
axes[0].legend()
plt.tight_layout()
plt.show()

### 6.3 - Exemples

Pour un modèle et une variante : trois photos qui gagnent le rang 1, trois qui le perdent. Chaque ligne montre la requête et le bon dauphin, bruts puis préparés, avec le rang avant → après.

In [ ]:
FOCUS = "cetaceanet"  # le modèle à regarder : cetaceanet, miewid, megadescriptor
# la variante : _detoure, _fond_blanc, _fond_blanc_marge3 ou _recadre_mer
FOCUS_VARIANT = "_detoure"
from pipeline.embeddings import crop_keep_sea  # noqa: E402

SEGMENTER = dorsalmatch.Segmenter()
# Préparation de chaque variante
PREPARE = {
    "_detoure": SEGMENTER.remove_background,
    "_fond_blanc": dorsalmatch.Segmenter({"crop": False}).remove_background,
    "_fond_blanc_marge3": dorsalmatch.Segmenter(
        {"crop": False, "margin": 0.03}
    ).remove_background,
    "_recadre_mer": lambda image: crop_keep_sea(SEGMENTER, image),
}


def rank_by_line(model):
    return {result["lines"][0]: result["rang"] for result in BY_PHOTO[model]}


def show_change(line, model, suffix):
    query = history[line]
    match = catalogue_photo(query["individu"], query["cote"])
    prepare, variant = PREPARE[suffix], VARIANTS[suffix]
    query_image = small(photo_path("historique", query["livraison"], query))
    match_image = small(photo_path("catalogue", CATALOGUE, match))
    images = [
        query_image,
        prepare(query_image),
        match_image,
        prepare(match_image),
    ]
    names = [
        "historique brute",
        f"historique · {variant}",
        "bon dauphin brut",
        f"bon dauphin · {variant}",
    ]
    before, after = rank_by_line(model)[line], rank_by_line(model + suffix)[line]
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
    for ax, image, name in zip(axes, images, names, strict=True):
        ax.imshow(image)
        ax.set_title(name, fontsize=9)
        ax.axis("off")
    fig.suptitle(
        f"{model} · {query['individu']} · {query['cote']} · rang {before} → {after}",
        x=0.01,
        ha="left",
    )
    plt.show()


changes = TRANSITIONS[(FOCUS, FOCUS_VARIANT)]
variant = VARIANTS[FOCUS_VARIANT]
won = changes.index[(changes["brut"] != CASES[0]) & (changes[variant] == CASES[0])]
lost = changes.index[(changes["brut"] == CASES[0]) & (changes[variant] != CASES[0])]
print(
    f"{FOCUS} · {variant} : {len(won)} photos gagnent le rang 1, {len(lost)} le perdent"
)
for title, lines in (("GAGNE", won), ("PERD", lost)):
    print(f"\n--- {title} le rang 1 ---")
    for line in RANDOM.choice(sorted(lines), size=min(3, len(lines)), replace=False):
        show_change(int(line), FOCUS, FOCUS_VARIANT)